In [67]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
  mean_absolute_error,
  mean_squared_error, 
  root_mean_squared_error, 
  r2_score,accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  confusion_matrix,
  classification_report,
  roc_auc_score)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, chi2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("Test.csv")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
print(df_train.head())
print("***************************************************")
print(df_train.info())
print("***************************************************")
print(df_train.describe())

In [ ]:
print("Is Null:")
fields_with_null_values = df_train.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_train.dtypes[fields_with_null_values.index]
})

print(null_info)

In [ ]:
df_train["LotFrontage"] = (
    df_train.groupby("Neighborhood")["LotFrontage"]
      .transform(lambda x: x.fillna(x.median()))
)

df_train["Electrical"] = df_train["Electrical"].fillna(
    df_train["Electrical"].mode()[0]
)

df_train["Alley"] = df_train["Alley"].fillna("None")
df_train["MasVnrType"] = df_train["MasVnrType"].fillna("None")
df_train["MasVnrArea"] = df_train["MasVnrArea"].fillna(0)
df_train["BsmtQual"] = df_train["BsmtQual"].fillna("None")
df_train["BsmtCond"] = df_train["BsmtCond"].fillna("None")
df_train["BsmtExposure"] = df_train["BsmtExposure"].fillna("None")
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].fillna("None")
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].fillna("None")
df_train["FireplaceQu"] = df_train["FireplaceQu"].fillna("None")
df_train["GarageType"] = df_train["GarageType"].fillna("None")
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna(0)
df_train["GarageFinish"] = df_train["GarageFinish"].fillna("None")
df_train["GarageQual"] = df_train["GarageQual"].fillna("None")
df_train["GarageCond"] = df_train["GarageCond"].fillna("None")
df_train["PoolQC"] = df_train["PoolQC"].fillna("None")
df_train["Fence"] = df_train["Fence"].fillna("None")
df_train["MiscFeature"] = df_train["MiscFeature"].fillna("None")

In [ ]:
print("Is Null:")
fields_with_null_values = df_test.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_test.dtypes[fields_with_null_values.index]
})

print(null_info)

In [ ]:
neighborhood_avg = [
  "LotFrontage"
]

str_category = [
  "MSZoning",
  "Utilities",
  "Exterior1st",
  "Exterior2nd",
  "SaleType"
]

na_to_none = [
  "Alley",
  "MasVnrType",
  "BsmtQual",
  "BsmtCond",
  "BsmtExposure",
  "BsmtFinType1",
  "BsmtFinType2",
  "KitchenQual",
  "Functional",
  "FireplaceQu",
  "GarageType",
  "GarageFinish",
  "GarageQual",
  "GarageCond",
  "PoolQC",
  "Fence",
  "MiscFeature"
]

na_to_0 = [
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtUnfSF",
  "TotalBsmtSF",
  "BsmtFullBath",
  "BsmtHalfBath",
  "MasVnrArea",
  "GarageYrBlt",
  "GarageCars",
  "GarageArea"
]

for item in neighborhood_avg:
  df_test[item] = (
    df_test.groupby("Neighborhood")[item]
    .transform(lambda x: x.fillna(x.median()))
  )

for item in str_category:
  df_test[item] = df_test[item].fillna(
      df_test[item].mode()[0]
  )

for item in na_to_none:
  df_test[item] = df_test[item].fillna("None")

for item in na_to_0:
  df_test[item] = df_test[item].fillna(0)

In [ ]:
# Creating Classification Dataset
classification_df_train = df_train.copy()

# 0 = Low, 1 = Medium, 2 = High
classification_df_train["PriceCategory"] = pd.qcut(
    classification_df_train["SalePrice"],
    q=3,
    labels=[0, 1, 2]
)

classification_df_train.drop(
    columns=["SalePrice"],
    inplace=True
)
classification_df_train["PriceCategory"].value_counts()


Nominal Catagories:
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"

Y/N:
"CentralAir"

Ordinal 1-10:
"OverallQual",
"OverallCond"

Ordinal Po-Ex(5):
"ExterQual",
"ExterCond",
"HeatingQC",
"KitchenQual"

Ordinal Na-Ex(6):
"BsmtQual",
"BsmtCond",
"FireplaceQu",
"GarageQual",
"GarageCond"

Ordinal Na-Ex(5):
"PoolQC"

Ordinal Misc:
"BsmtExposure",
"BsmtFinType1",
"BsmtFinType2",
"Functional",
"GarageFinish",
"PavedDrive"??



In [ ]:
#Ordinal Maps
datasets = [
  df_train,
  classification_df_train,
  df_test
]

for dataset in datasets:
  # Yes/No Map
  quality_map = {
    "N": 0,
    "Y": 1
  }

  dataset["CentralAir"] = dataset["CentralAir"].map(quality_map)

  # Poor - Excellent 5 Rankings Map
  quality_map = {
    "Po": 0,
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_po_ex_5 = [
  "ExterQual",
  "ExterCond",
  "HeatingQC",
  "KitchenQual"
  ]

  for category in ordinal_po_ex_5:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 6 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
  }

  ordinal_na_ex_6 = [
  "BsmtQual",
  "BsmtCond",
  "FireplaceQu",
  "GarageQual",
  "GarageCond"
  ]

  for category in ordinal_na_ex_6:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 5 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_na_ex_5 = [
    "PoolQC"
  ]

  for category in ordinal_na_ex_5:
    dataset[category]=dataset[category].map(quality_map)


  # Misc. Ordinal Rankings Map

  # "BsmtExposure"
  quality_map ={
    "NA": 0,
    "None": 0,
    "No": 0,
    "Mn":	2,
    "Av":	3,
    "Gd":	4
  }

  dataset["BsmtExposure"]=dataset["BsmtExposure"].map(quality_map)

  # "BsmtFinType1"
  # "BsmtFinType2"
  quality_map = {
    "NA": 0,
    "None": 0,
    "No": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
  }

  dataset["BsmtFinType1"]=dataset["BsmtFinType1"].map(quality_map)
  dataset["BsmtFinType2"]=dataset["BsmtFinType2"].map(quality_map)

  # "Functional"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
  }
  dataset["Functional"]=dataset["Functional"].map(quality_map)

  # "GarageFinish"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
  }

  dataset["GarageFinish"]=dataset["GarageFinish"].map(quality_map)


  # "PavedDrive"
  quality_map ={
    "N": 0,
    "NA": 0,
    "None": 0,
    "P" : 1,
    "Y": 2
  }

  dataset["PavedDrive"]=dataset["PavedDrive"].map(quality_map)

In [ ]:
# Regression dataset targeting sale price
X = df_train.drop(columns=["SalePrice"])
y=df_train["SalePrice"]



# Classification dataset targeting price category 
y_class = classification_df_train["PriceCategory"]
X_class = classification_df_train.drop(columns=["PriceCategory"])

X_test = df_test

In [ ]:
#Nominal Maps
nominal_columns = [
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
        OneHotEncoder(handle_unknown="ignore", drop="first"),
        nominal_columns)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(X)
X_class = preprocessor.transform(X_class)
X_test = preprocessor.transform(X_test)

In [ ]:
# Regression Split
X_train, X_val, y_train, y_val = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=42
)

# Classification Split
X_class_train, X_class_val, y_class_train, y_class_val = train_test_split(
  X_class,
  y_class,
  test_size=0.2,
  random_state=42
)

# Create Results List
results = []

In [ ]:
# Chi-Square Feature Selection - Classification Dataset - (Running before scaling to avoid negatives)
selector = SelectKBest(
  score_func=chi2,
  k=20
)

X_class_train_chi2 = selector.fit_transform(X_class_train, y_class_train)
X_class_val_chi2 = selector.transform(X_class_val)

In [ ]:
# Scale regression dataset
reg_scaler = StandardScaler()
X_train = reg_scaler.fit_transform(X_train)
X_val = reg_scaler.transform(X_val)

# Scale classification dataset
class_scaler = StandardScaler()
X_class_train = class_scaler.fit_transform(X_class_train)
X_class_val = class_scaler.transform(X_class_val)

# Scale Chi-Square Classification Dataset
chi2_scaler = StandardScaler()
X_class_train_chi2 = chi2_scaler.fit_transform(X_class_train_chi2)
X_class_val_chi2 = chi2_scaler.transform(X_class_val_chi2)


# Fix classification dataset label type
y_class_train = y_class_train.astype("int32")
y_class_val = y_class_val.astype("int32")

In [ ]:
# PCA Feature Selection - Regression DataSet
pca = PCA(n_components=0.95)

X_train_PCA = pca.fit_transform(X_train)
X_val_PCA = pca.transform(X_val)


In [ ]:
# ANOVA F-Test Feature Selection - Regression & Classification Datasets
selector = SelectKBest(
  score_func=f_regression,
  k=20
)

X_train_ANOVA = selector.fit_transform(X_train, y_train)
X_val_ANOVA = selector.transform(X_val)

X_class_train_ANOVA = selector.fit_transform(X_class_train, y_class_train)
X_class_val_ANOVA = selector.transform(X_class_val)

In [ ]:
# Regression Model - Linear Regression
model = LinearRegression()

# Full feature set
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  


# PCA Selected Feature Set
model.fit(X_train_PCA, y_train)

y_pred = model.predict(X_val_PCA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "PCA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

# ANOVA Selected Feature Set
model.fit(X_train_ANOVA, y_train)

y_pred = model.predict(X_val_ANOVA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

In [ ]:
# Classical Classification Model - Logistical Regression
model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model.fit(X_class_train, y_class_train)

y_pred = model.predict(X_class_val)
y_proba = model.predict_proba(X_class_val)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# ANOVA Selected Feature Set

model.fit(X_class_train_ANOVA, y_class_train)

y_pred = model.predict(X_class_val_ANOVA)
y_proba = model.predict_proba(X_class_val_ANOVA)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Chi-Squared Selected Feature Set
model.fit(X_class_train_chi2, y_class_train)

y_pred = model.predict(X_class_val_chi2)
y_proba = model.predict_proba(X_class_val_chi2)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "Chi Squared Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

In [ ]:
# Deep Learning Regression Model - Feedforward Dense Neural Network - Full Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_pred = regressor.predict(X_val).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  


# Deep Learning Regression Model - Feedforward Dense Neural Network - PCA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_PCA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_PCA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_pred = regressor.predict(X_val_PCA).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "PCA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

# Deep Learning Regression Model - Feedforward Dense Neural Network - ANOVA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_ANOVA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_pred = regressor.predict(X_val_ANOVA).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
}) 

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 39144726528.0000 - mae: 181520.7500 - val_loss: 37836365824.0000 - val_mae: 181088.0625
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39132590080.0000 - mae: 181493.7344 - val_loss: 37815230464.0000 - val_mae: 181041.4219
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39091015680.0000 - mae: 181405.2344 - val_loss: 37752328192.0000 - val_mae: 180906.3438
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38985736192.0000 - mae: 181185.7656 - val_loss: 37610377216.0000 - val_mae: 180602.2500
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38772850688.0000 - mae: 180741.0469 - val_loss: 37349638144.0000 - val_mae: 180043.4219
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38406647808.0000 - mae: 179974.6094 - val_loss: 36927459328.0000 - val_mae: 179133.8125
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 37853630464.0000 - mae: 178809.6250 - val_loss: 36313391104.0000 - val_

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 39145349120.0000 - mae: 181521.9219 - val_loss: 37838491648.0000 - val_mae: 181092.1406
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39139975168.0000 - mae: 181507.5469 - val_loss: 37830828032.0000 - val_mae: 181072.3438
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39127592960.0000 - mae: 181477.1406 - val_loss: 37815189504.0000 - val_mae: 181033.6406
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39102697472.0000 - mae: 181417.7812 - val_loss: 37785931776.0000 - val_mae: 180964.5312
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39057604608.0000 - mae: 181314.9844 - val_loss: 37737345024.0000 - val_mae: 180853.1250
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38984663040.0000 - mae: 181153.0000 - val_loss: 37662732288.0000 - val_mae: 180686.7812
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38874333184.0000 - mae: 180913.1406 - val_loss: 37557358592.0000 - val_

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 39146184704.0000 - mae: 181524.4375 - val_loss: 37840146432.0000 - val_mae: 181096.9531
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39142162432.0000 - mae: 181514.6875 - val_loss: 37833203712.0000 - val_mae: 181080.3281
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39129251840.0000 - mae: 181485.0156 - val_loss: 37813219328.0000 - val_mae: 181034.0469
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39097778176.0000 - mae: 181413.5312 - val_loss: 37770129408.0000 - val_mae: 180934.7188
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39038160896.0000 - mae: 181276.7031 - val_loss: 37693317120.0000 - val_mae: 180758.0312
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38934843392.0000 - mae: 181043.8125 - val_loss: 37572792320.0000 - val_mae: 180479.7031
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38780805120.0000 - mae: 180692.6719 - val_loss: 37395013632.0000 - val_

[{'Model Type': 'Deep Learning Regression',
  'Model': 'Dense Neural Network Regression',
  'Feature Set': 'Full Feature Set',
  'Output': 'Predicted SalePrice',
  'MAE': 29253.72265625,
  'MSE': 1940664192.0,
  'RMSE': 44052.96875,
  'R2': 0.7469906806945801,
  'Accuracy': None,
  'Precision': None,
  'Recall': None,
  'F1': None,
  'AUC': None},
 {'Model Type': 'Deep Learning Regression',
  'Model': 'Dense Neural Network Regression',
  'Feature Set': 'PCA Selected Feature Set',
  'Output': 'Predicted SalePrice',
  'MAE': 25819.7421875,
  'MSE': 1402561920.0,
  'RMSE': 37450.79296875,
  'R2': 0.8171443939208984,
  'Accuracy': None,
  'Precision': None,
  'Recall': None,
  'F1': None,
  'AUC': None},
 {'Model Type': 'Deep Learning Regression',
  'Model': 'Dense Neural Network Regression',
  'Feature Set': 'ANOVA Selected Feature Set',
  'Output': 'Predicted SalePrice',
  'MAE': 46775.12890625,
  'MSE': 3606739712.0,
  'RMSE': 60056.13671875,
  'R2': 0.5297801494598389,
  'Accuracy': No

In [ ]:
# Deep Learning Classification - Dense Neural Network Classification - Full Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_proba = model.predict(X_class_val)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Deep Learning Classification - Dense Neural Network Classification - ANOVA Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_ANOVA,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_proba = model.predict(X_class_val_ANOVA)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Deep Learning Classification - Dense Neural Network Classification - Chi-Squared Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_chi2.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_chi2,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

y_proba = model.predict(X_class_val_chi2)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5846 - loss: 0.9014 - val_accuracy: 0.7521 - val_loss: 0.5792
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8287 - loss: 0.4415 - val_accuracy: 0.8077 - val_loss: 0.4505
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8951 - loss: 0.3123 - val_accuracy: 0.8120 - val_loss: 0.4218
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9261 - loss: 0.2332 - val_accuracy: 0.8120 - val_loss: 0.4242
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9454 - loss: 0.1862 - val_accuracy: 0.8291 - val_loss: 0.4065
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9625 - loss: 0.1512 - val_accuracy: 0.8376 - val_loss: 0.4017
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9786 - loss: 0.1105 - val_accuracy: 0.8376 - val_loss: 0.4032
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9861 - loss: 0.0884 - val_accuracy: 0.8462 - val_loss: 0.4

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6767 - loss: 0.7248 - val_accuracy: 0.7821 - val_loss: 0.5420
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.5188 - val_accuracy: 0.8162 - val_loss: 0.4624
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8051 - loss: 0.4622 - val_accuracy: 0.8291 - val_loss: 0.4323
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8298 - loss: 0.4252 - val_accuracy: 0.8333 - val_loss: 0.4183
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8308 - loss: 0.4026 - val_accuracy: 0.8162 - val_loss: 0.4133
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8448 - loss: 0.3801 - val_accuracy: 0.8462 - val_loss: 0.3817
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8544 - loss: 0.3644 - val_accuracy: 0.8547 - val_loss: 0.3771
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8630 - loss: 0.3496 - val_accuracy: 0.8376 - val_loss: 0.3

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5921 - loss: 0.8577 - val_accuracy: 0.6795 - val_loss: 0.7028
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7002 - loss: 0.6442 - val_accuracy: 0.7137 - val_loss: 0.6444
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7409 - loss: 0.5817 - val_accuracy: 0.7350 - val_loss: 0.6337
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7570 - loss: 0.5494 - val_accuracy: 0.7521 - val_loss: 0.6094
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7602 - loss: 0.5313 - val_accuracy: 0.7564 - val_loss: 0.6066
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7827 - loss: 0.5070 - val_accuracy: 0.7607 - val_loss: 0.6192
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7912 - loss: 0.4892 - val_accuracy: 0.7564 - val_loss: 0.6070
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7901 - loss: 0.4741 - val_accuracy: 0.7564 - val_loss: 0.6

[{'Model Type': 'Deep Learning Classification',
  'Model': 'Dense Neural Network Classification',
  'Feature Set': 'Full Feature Set',
  'Output': 'Predicted PriceCategory',
  'MAE': None,
  'MSE': None,
  'RMSE': None,
  'R2': None,
  'Accuracy': 0.8013698630136986,
  'Precision': 0.8017602864897156,
  'Recall': 0.8013698630136986,
  'F1': 0.8008427421884896,
  'AUC': 0.9427615288060495},
 {'Model Type': 'Deep Learning Classification',
  'Model': 'Dense Neural Network Classification',
  'Feature Set': 'ANOVA Selected Feature Set',
  'Output': 'Predicted PriceCategory',
  'MAE': None,
  'MSE': None,
  'RMSE': None,
  'R2': None,
  'Accuracy': 0.821917808219178,
  'Precision': 0.8195550339921939,
  'Recall': 0.821917808219178,
  'F1': 0.8204649623936985,
  'AUC': 0.9296315981448939},
 {'Model Type': 'Deep Learning Classification',
  'Model': 'Dense Neural Network Classification',
  'Feature Set': 'Full Feature Set',
  'Output': 'Predicted PriceCategory',
  'MAE': None,
  'MSE': None,
  

In [ ]:
results_df = pd.DataFrame(results)
results_df